In [1]:
import yaml
import numpy as np 
import pandas as pd
from pytorch_lightning import Trainer
import importlib

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = 'config/commonvoice/cv_word_baseline.yaml'
config = yaml.load(open(path, 'r'), Loader=yaml.FullLoader)

In [3]:
!nvidia-smi

Tue May 30 09:24:10 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 460.106.00   Driver Version: 460.106.00   CUDA Version: 11.2     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  GeForce GTX 108...  On   | 00000000:02:00.0 Off |                  N/A |
| 23%   33C    P8     9W / 250W |      0MiB / 11178MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [4]:
config['loader']['num_workers'] = 0
config['loader']['batch_size'] = 32
config['corpora_name'] = 'TIMIT'
config['audio']['rep_kwargs']['rep_on_gpu'] = True
config['corpus']['root']  = '/om2/user/imgriff/datasets/timit/clean_timit_targets_attn_task_0.1rms.pdpkl'

ckpt_path =  "attn_cue_models/cv_baseline_word_task/checkpoints/epoch=8-step=18416-v2.ckpt"


In [5]:
# sys.path.append('../')
from src import cv_word_lightning
importlib.reload(cv_word_lightning)

CommonVoiceWordRec = cv_word_lightning.CommonVoiceWordRec

module = CommonVoiceWordRec.load_from_checkpoint(checkpoint_path=ckpt_path, config=config)


Eval on TIMIT stimuli
center_crop=True
binaural=False
using FIR cochleagram


In [6]:
import pickle 
class_map = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_word_int_label_dict.pkl", "rb" )) 


In [7]:
from corpus import timit 
importlib.reload(timit)
eval_stim_path = '/om2/user/imgriff/datasets/timit/clean_timit_targets_attn_task_0.1rms.pdpkl'
dataset = timit.TIMIT_CV_Compat_Prepaired(eval_stim_path, clean_targets=True)

# timit_to_cv_mappting: dataset.class_remap

cv_remap = dataset.class_remap

In [8]:
## Get wsn data as dataframe 

wsn_eval_df = pd.read_pickle("/om2/user/msaddler/spatial_audio_pipeline/assets/wsn/manifest_JSIN_behavioral_unique_speech.pdpkl")

In [9]:
wsn_eval_df

,client_id,clip_dur_in_s,clip_end_in_s,clip_start_in_s,corpus,corpus_int,label_talker_int,label_word_int,path,split,split_int,sr,src_fn,talker_int,total_file_duration_in_s,word,word_int
0,46e,0.50,4.76,4.26,wsj,2,14,644,/home/raygon/projects/user/jfeather/jsinDatase...,eval,2,16000,/om/data/public/wall-street-journal/csr_2_comp...,13,10.525500,shares,643
30,ljl,0.35,251.48,251.13,swc,1,227,419,/home/raygon/deepFerret/data/sources/speech/sw...,eval,2,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,226,282.393832,meaning,418
60,viktor-o-ledenyov,0.47,1246.34,1245.87,swc,1,308,681,/home/raygon/deepFerret/data/sources/speech/sw...,eval,2,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,307,2069.366712,stated,680
90,thevoicebeforethevoid,0.29,18.86,18.57,swc,1,296,51,/home/raygon/deepFerret/data/sources/speech/sw...,eval,2,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,295,1425.587664,april,50
120,willmcw,0.37,798.57,798.20,swc,1,317,271,/home/raygon/deepFerret/data/sources/speech/sw...,eval,2,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,316,1739.858141,following,270
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23640,batwoodman,0.41,2090.46,2090.05,swc,1,160,512,/home/raygon/deepFerret/data/sources/speech/sw...,eval,2,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,159,7148.244172,playing,511
23670,46y,0.35,4.05,3.70,wsj,2,30,222,/home/raygon/projects/user/jfeather/jsinDatase...,eval,2,16000,/om/data/public/wall-street-journal/csr_2_comp...,29,6.939375,effect,221
23700,salsasam,0.37,1193.74,1193.37,swc,1,271,673,/home/raygon/deepFerret/data/sources/speech/sw...,eval,2,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,270,3334.353560,spread,672
23730,gonzonoir,0.65,461.42,460.77,swc,1,198,634,/home/raygon/deepFerret/data/sources/speech/sw...,eval,2,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,197,1206.266485,september,633


In [10]:
import torchmetrics
import librosa 
from tqdm import tqdm
import torch

In [11]:
## Just loop through in vanilla pytorch 
accuracy = torchmetrics.Accuracy().cuda()

results = []

output_rate = config['audio']['rep_kwargs']['sr']
audio_dur_s = 2 # in seconds 

audio_transforms = module.audio_transforms
model = module.model.eval().cuda()

for row in tqdm(wsn_eval_df.itertuples(), total=len(wsn_eval_df)):

    aud_buffer = (audio_dur_s - row.clip_dur_in_s) / 2 
    start = row.clip_start_in_s - aud_buffer
    audio, _ = librosa.load(row.src_fn, sr=output_rate,
                            offset=start,
                            duration = audio_dur_s,
                            res_type='soxr_vhq')

    audio, _  = audio_transforms(audio, None)
    # get int label in cv vocab classes 
    label = torch.from_numpy(np.array(cv_remap[row.label_word_int])).unsqueeze(0).cuda()
    

    output = model.forward(audio.cuda(), None) 
    model_class_guess = output.softmax(-1).argmax(-1).cpu().item()

    accuracy(output, label)

    results.append({
        "model_label": model_class_guess,
        "model_word": dataset.cv_class_map[model_class_guess],
        "label": label.cpu().item(),
        "word": row.word
    })



100%|██████████| 793/793 [02:23<00:00,  5.53it/s]


In [12]:
row.word

'direct'

In [13]:
dataset.cv_class_map[output.softmax(-1).argmax(-1).item()]

'direct'

In [14]:
dataset.cv_class_map[cv_remap[row.label_word_int]]

'direct'

In [15]:
## Global acc with oov labels
print(accuracy.compute())

tensor(0.3544, device='cuda:0')


In [16]:

model_results = pd.DataFrame.from_records(results)

In [17]:
(model_results['model_label'] == model_results['label']).mean()

0.35435056746532156

In [18]:
wsn_cv_classes = [cv_remap[class_int] for class_int in wsn_eval_df.label_word_int.values]

In [ ]:
model_results["cv_label"] = wsn_cv_classes

In [19]:
valid_results = model_results[model_results.label !=0]
valid_results['accuracy'] = valid_results['model_label'] == valid_results['label']


/tmp/ipykernel_4065/1339513799.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_results['accuracy'] = valid_results['model_label'] == valid_results['label']


In [20]:

print(f"N examples: {len(valid_results)}")
print(f"Mean accuracy: {valid_results.accuracy.mean():.3}")
print(f"Standard error: {valid_results.accuracy.sem():.3}")

N examples: 437
Mean accuracy: 0.643
Standard error: 0.0229
